# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [20]:
import subprocess
result = subprocess.run(
    ['pip', 'show', 'langchain-community'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [21]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os
import sys 
print (sys.executable)
print (sys.path)

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/home/sajalosta/DrGreggCourse/code/AIEC_v1.0/01_Dense_Vector_Retrieval/.venv/bin/python
['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/sajalosta/DrGreggCourse/code/AIEC_v1.0/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages']


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [22]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [23]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [24]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [25]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [26]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:
✈️🧑‍🚀🚀👨‍🚀
1. It gives us information about the length of the document which can help us to compare if the loader actually loaded all the pages of the document. IT can be a useful tool to help prevent data loss.
2. You can filter the vector database on the basis of different fields of the metadata.
3. Displaying information about the source of the data in your output can help with transparency that builds trust.
4. It helps us to debug when there is hallucination. There are times when the output drifts due to different dataset revisions and the responses can be hallucinated. With metadata it helps us to compare the hallucinated version with another version that didn't hallucinate and this gives us a big clue for troubleshooting hallucination.
5. Also, we will know about the source of the data through metadata and hence we can go to the exact document and page/paragrpah ourselves for more information
✈️✈️🎆🎆

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [108]:
chunk_size = 1500
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 90 chunks.
Chunk size: 1500 characters
Chunk overlap: 200 characters


In [109]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer: 
                 
1.Smaller the overlap, or no overlap, would mean that all the context at the chunk break hasn't been captured and the response probably would not make sense or can hallucinate.  ⛹️⛹️

2.Bigger the overlap would mean that we are adding redundancy into the chunks and exhaust the token limit faster making the application expensive.
🍿🧂
 
3.Smaller the chunks for a given document would mean more latency for storage and retrieval from the vector DB as we would occupy more space in the vector DB. 🤦‍♀️🤦‍♂️🤷‍♀️

4.Bigger the chunk size could mean that we are feeding more info to the LLM than it needs and there is a chance it doesn't pick all the needed info from the chunks results in insufficient data retrieval.  🍿🧂

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [110]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [111]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [112]:
retrieval_k = 4
retrieval_query = " My cat is getting cough and can't sleep, what do you think is going on?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.446 | page=7 | start_index=1326
relevant. The physical examination for k ittens typically focuses on detection of congenital issues such as a heart murmur, hernia, or cleft palate. A detailed oral examination is performed to detect abnormalities of dentition. The use of fecal scoring charts is very helpful to ensure that the client can accurately identify stool consistency. 25,26
--------------------------------------------------------------------------------
Source 2 | score=0.414 | page=8 | start_index=3963
most subtle signs of anxiety and tension. Clinical signs of fear or stress in cats are displayed through characteristic body postures, vocalizations, and activity. A cowering (tense, ﬂattened) position where the head is lower than the body may be indicative of stress or fear in cats. A state of distress may also be characterized by crouching, crawl
--------------------------------------------------------------------------------
Source 3 | score=0.409 | page=8 | 

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

A) Similarity score helps me to understand the degree to which two vectors (sentence vectors , document vectors, word vectors) are similar to each other contextually (it may not convey similarity in the exact meaning).
For example, king and queen are contextually similar in their usage with a high similarity score than king vs banana, but (king, queen) are not the same in their meaning. ⛹️‍♀️⛹️‍♂️⛹️

king <> queen                  score=0.591

king <> banana                 score=0.310

B) However, the score by itself does not prove anything unless it is compared to others scores in the Vector DB. 😒 So we rank the similairty scores and that gives us an idea of the most contextually similarity scores 🎇🎆 
                



## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [113]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [117]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    ##print(context)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [118]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 7, score 0.446
relevant.
The physical examination for k ittens typically focuses on
detection of congenital issues such as a heart murmur, hernia, or
cleft palate. A detailed oral examination is performed to detect
abnormalities of dentition. The use of fecal scoring charts is very
helpful to ensure that the client can accurately identify stool
consistency.
25,26
Young Adult Cats
Lower airway disease is common in young adult cats.
27 Coughing is a
typical sign of feline bronchial disease; however, the veterinarian
must consider the role of heartworm-associated respiratory disease
(HARD), transtracheal migration of roundworm ( Toxocara cati ),
and lungworm. Asking speci ﬁc questions regarding the presence of
coughing is helpful for early diagnosis and treatment. Coughing is
not typically a hallmark of cardiac disease in cats, in contrast to
canine patients, nor is it caused by hairballs. Y oung adult cats de-
veloping cardiac conditions such as

In [119]:
answer_k = 4

result = answer_question(
    "what are diseases and conditions that are seen in young adult cat 1-6 years of age?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

For **young adult cats (1–6 years)**, the guideline highlights these diseases/conditions as important to consider:

- **Lower airway disease** [Source 2]
- **Feline bronchial disease** (coughing is a typical sign) [Source 2]
- **Heartworm-associated respiratory disease (HARD)** [Source 2]
- **Transtracheal migration of roundworm (*Toxocara cati*)** [Source 2]
- **Lungworm** [Source 2]
- **Cardiac conditions such as hypertrophic cardiomyopathy** (may be asymptomatic or show reduced activity/exercise tolerance) [Source 2]
- **Chronic enteropathy or gastrointestinal (GI) disease** (suggested by vomiting, vomiting hairballs, diarrhea, or weight loss) [Source 2]

If you’re asking about a specific cat’s symptoms, I’d recommend contacting a veterinarian for an evaluation.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 7, 'start_index': 2647, 'score': 0.6632575565568581}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 7, 'start_ind

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [17]:
vibe_check_questions = [
    "What are the different life stages of a cat",
    "What are some of the diseases commonly seen in young adult cats?",
    "give me information listed in table 3 titled 'Diseases and conditions that require particular focus' "
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What are the different life stages of a cat
The guidelines define four age-related life stages for cats: kitten (birth to 1 year), young adult (1 to 6 years), mature adult (7 to 10 years), and senior (over 10 years). There is also a fifth end-of-life stage that can occur at any age [Source 2].
Question: What are some of the diseases commonly seen in young adult cats?
Some commonly seen diseases/conditions in young adult cats mentioned in the guideline include lower airway disease, feline bronchial disease, heartworm-associated respiratory disease (HARD), roundworm migration (Toxocara cati), lungworm, and cardiac conditions such as hypertrophic cardiomyopathy [Source 2]. The guideline also notes vomiting, vomiting hairballs, diarrhea, and chronic GI signs as important concerns to watch for, though these are signs rather than specific diseases [Source 2].
Question: give me information listed in table 3 titled 'Diseases and conditions that require particular focus' 
I don’t have

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

Story goes:

Round 1 of experimentations!!!!! ✈️👨‍🚀🚀🧑‍🚀🪶
So I played around with chunk size (1000) and overlap (200). For the question, "diseases commonly seen in young adult cats",  the retrieved context didn't contain information for the query and so was irrelevant. Even after increasing the no. of chunks to 4, the context was not relevant.

Round 2 of experimentation!!!!!! 🍿🍿🍿🍿🍿
Increased the chunk size to 1500, even reduced chunk size back to 2. Now the relevant context was relevant. The generation was pretty crisp and accurate.

Conclusions 🚕🚕🚕🛺🛺🚒
1. You should experiment with chunk size/ no. of retrieved chunks (k) to ensure you are seeing accurate results and not just assume that the retrieved context will always be relevant.
2. Also, when the LLM was prompted about context in Table 3 where the data was inside a table, the relevant context couldn't pull the data from the table and so the generation said it doesn't have required information. Maybe using another loader would be useful for reading tabular data.

So I saw that the retrieved context either did contain relevant context or didn't based on experimentation above and on the basis of the limitations of the loader. 🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️




## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [18]:
# Activity workspace
# Try retrieval tuning here.
vibe_check_questions = [
    "What are some of the diseases commonly seen in young adult cats?"
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)
### YOUR CODE HERE

Question: What are some of the diseases commonly seen in young adult cats?
Some commonly seen diseases/conditions in young adult cats mentioned in the guideline are lower airway disease, feline bronchial disease, heartworm-associated respiratory disease (HARD), roundworm migration (Toxocara cati), lungworm, and cardiac conditions such as hypertrophic cardiomyopathy [Source 2].


### 🏗️ Activity Notes

I answered this question above but repeating here for completeness 👨‍🚀👨‍🚀🛺🛺
- Setting changed:  Changed chunk size to 1500 
- Before:           chunk size was 1000.  LLM said it doesn't have information for the query "What are some of the diseases commonly seen in young adult cats?",
- After:            After increasing chunk size to 1500, reducing chunk size to 2 the LLM gave correct and accurate response!
- Did retrieval improve? Why or why not?   Yes retrieval became accurate after playing around with the 3 parameters above.